# YOLO 2회차 추가 실습 — 위험구역 침입 감시 시스템

CCTV 영상에서 사람을 추적하고, **화면 위에 직접 그린 출입 금지 구역**에 누가 들어갔는지, 얼마나 머물렀는지 감시하는 시스템을 만듭니다.
마지막에는 사람들이 실제로 어디로 다니는지 **동선 히트맵**으로 보고 금지 구역 위치가 적절했는지도 따져 봅니다.

| 단계 | 만드는 것 | 결과물 |
|---|---|---|
| Step 1 | 사람만 탐지하기 | 프레임별 인원 그래프 |
| Step 2 | 사람 추적 + 발 위치 기록 | 관측 표 `obs` |
| Step 3 | 금지 구역 그리기 + 침입 판정 | 금지 구역 좌표, 구역 안 인원 |
| Step 4 | 진입·이탈 이벤트와 체류 시간 경고 | `zone_events.csv`, `zone_monitor.mp4` |
| Step 5 | 동선 히트맵과 감시 리포트 | `heatmap.png`, `zone_report.png` |

**실습 영상**: OpenCV 공개 샘플 영상 `vtest.avi` (건물 앞 광장을 비추는 고정 카메라, 768×576, 10fps, 79.5초)

**실행 환경**: Google Colab, 런타임 유형 **T4 GPU** (CPU에서도 동작하지만 느립니다)

## 0. 준비

In [ ]:
!pip -q install ultralytics lap

In [ ]:
import os, json, shutil, base64, subprocess
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import HTML, display
from ultralytics import YOLO

print('GPU 사용 가능:', torch.cuda.is_available())

try:
    from google.colab import drive, output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    drive.mount('/content/drive')
    SAVE_DIR = Path('/content/drive/MyDrive/yolo_day2/zone')
else:
    SAVE_DIR = Path('outputs/zone')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print('결과 저장 위치:', SAVE_DIR.resolve())

In [ ]:
def show_bgr(img, title=None, size=(9, 6)):
    plt.figure(figsize=size)
    plt.imshow(img[..., ::-1]); plt.axis('off')
    if title:
        plt.title(title)
    plt.show()

def show_video(path, width=640):
    h264 = str(Path(path).with_name(Path(path).stem + '_h264.mp4'))
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(path),
                    '-vcodec', 'libx264', '-pix_fmt', 'yuv420p', h264], check=True)
    data = base64.b64encode(open(h264, 'rb').read()).decode()
    display(HTML(f'<video width="{width}" controls loop autoplay muted>'
                 f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))

def save_out(path):
    dst = SAVE_DIR / Path(path).name
    if Path(path).resolve() != dst.resolve():
        shutil.copy(path, dst)
    return dst

In [ ]:
# 실습 영상 내려받기 (OpenCV 저장소의 공개 샘플 영상)
VIDEO = 'vtest.avi'
if not Path(VIDEO).exists():
    !wget -q https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi -O {VIDEO}

cap = cv2.VideoCapture(VIDEO)
FPS = cap.get(cv2.CAP_PROP_FPS) or 10
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
ok, FIRST = cap.read()
cap.release()
print(f'{VIDEO}: {W}x{H}, {FPS:.0f}fps, {N_FRAMES}프레임, {N_FRAMES / FPS:.1f}초')
show_bgr(FIRST, 'first frame')

---
## Step 1. 사람만 탐지하기

COCO 사전학습 모델(`yolo26n.pt`)은 80개 클래스를 찾습니다. 여기서는 **사람(클래스 0번)만** 필요하므로 `classes=[0]`으로 나머지는 처음부터 걸러냅니다. 후처리에서 거르는 것보다 빠르고 코드도 간단합니다.

In [ ]:
model = YOLO('yolo26n.pt')
PERSON = 0
CONF = 0.3

r = model(FIRST, classes=[PERSON], conf=CONF, verbose=False)[0]
print('첫 프레임에서 찾은 사람 수:', len(r.boxes))
show_bgr(r.plot(), 'person detection (first frame)')

In [ ]:
# 영상 전체 — 프레임마다 사람 수 세기
counts = [len(r.boxes) for r in model(VIDEO, stream=True, classes=[PERSON], conf=CONF, verbose=False)]
t = np.arange(len(counts)) / FPS
plt.figure(figsize=(10, 3))
plt.plot(t, counts, lw=1)
plt.xlabel('time (s)'); plt.ylabel('people in frame'); plt.title('People detected per frame')
plt.grid(alpha=.3); plt.show()
print(f'프레임당 평균 {np.mean(counts):.1f}명, 최대 {max(counts)}명')

**확인해 보기** — 그래프가 들쭉날쭉한 이유는 무엇일까요? 사람이 가려지거나, 화면 가장자리에서 일부만 보일 때 탐지가 끊깁니다. 이런 끊김은 Step 2의 트래킹이 메워 줍니다.

---
## Step 2. 사람 추적과 발 위치 기록

### 왜 '발 위치'인가

금지 구역은 **바닥** 위의 영역입니다. 사람 박스의 중심점은 허리 높이에 있어서, 카메라가 비스듬히 내려다보면 실제로 서 있는 곳보다 화면 위쪽에 찍힙니다.
그래서 박스의 **아래쪽 가운데 점(발 위치)** 을 그 사람이 서 있는 바닥 위치로 씁니다.

```
   ┌──────┐
   │  🧍  │   ← 박스 중심 (허리) — 바닥 위치로 쓰면 오차가 큼
   │      │
   └──●───┘   ← 발 위치 = ((x1 + x2) / 2, y2)
```

### 한 번 추적하고 표로 남기기

영상을 한 번만 추적하면서 **프레임마다, 사람마다** 한 줄씩 기록합니다. 뒤 단계(구역 판정, 체류 시간, 히트맵)는 모두 이 표로 계산하므로 모델을 다시 돌릴 필요가 없습니다.

In [ ]:
def track_people(src):
    rows = []
    for i, r in enumerate(model.track(src, stream=True, classes=[PERSON], conf=CONF,
                                      tracker='bytetrack.yaml', verbose=False)):
        if r.boxes.id is None:
            continue
        for (x1, y1, x2, y2), tid, cf in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.id.int().cpu().tolist(),
                                             r.boxes.conf.cpu().numpy()):
            rows.append({'frame': i, 'time_s': round(i / FPS, 2), 'track_id': tid,
                         'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                         'foot_x': (x1 + x2) / 2, 'foot_y': y2, 'conf': round(float(cf), 3)})
    return pd.DataFrame(rows)

obs = track_people(VIDEO)
print('관측 행 수:', len(obs), '| 추적된 사람(ID) 수:', obs['track_id'].nunique())
obs.round(1).head()

In [ ]:
# ID별로 몇 초 동안 보였는지
life = obs.groupby('track_id').agg(first_s=('time_s', 'min'), last_s=('time_s', 'max'), frames=('frame', 'count'))
life['seen_s'] = (life['frames'] / FPS).round(1)
print('5초 이상 추적된 ID:', int((life['seen_s'] >= 5).sum()), '/', len(life))
life.sort_values('seen_s', ascending=False).head(8)

In [ ]:
# 발 위치 궤적 그리기 — ID마다 다른 색
def id_color(tid):
    rng = np.random.default_rng(int(tid))
    return tuple(int(c) for c in rng.integers(60, 255, 3))

canvas = FIRST.copy()
for tid, g in obs.groupby('track_id'):
    pts = g[['foot_x', 'foot_y']].to_numpy().astype(np.int32)
    cv2.polylines(canvas, [pts], False, id_color(tid), 2)
show_bgr(canvas, 'foot trajectories of every tracked person')

**확인해 보기** — 궤적이 중간에 끊기고 다른 색으로 이어지는 곳이 있나요? 사람이 서로 겹치거나 가로등 기둥에 가려질 때 **ID가 바뀌는 현상(ID switch)** 입니다. Step 4에서 체류 시간을 잴 때 이 현상이 어떤 영향을 줄지 생각해 두세요.

---
## Step 3. 금지 구역 그리기와 침입 판정

### 3-1. 마우스로 금지 구역 그리기

아래 셀을 실행하면 첫 프레임이 나타납니다. **금지 구역의 꼭짓점을 차례로 클릭**하고 `완료`를 누르세요. 점은 3개 이상이어야 하고, 잘못 찍었으면 `되돌리기`를 누릅니다.

- 예시 시나리오: 화면 아래쪽 **잔디밭을 '장비 작업 중 출입 금지 구역'** 으로 정해 봅니다.
- Colab이 아닌 환경이거나 그리지 않고 넘어가면 미리 정해 둔 기본 구역(`DEFAULT_ZONE`)을 씁니다.
- **'모두 실행'으로 돌리면 그리기를 기다리지 않고 다음 셀이 실행되어 기본 구역이 쓰입니다.** 구역을 그린 뒤에는 다음 셀부터 다시 실행하세요. 구역을 다시 그렸을 때도 마찬가지입니다.

In [ ]:
DEFAULT_ZONE = [[0, 330], [330, 350], [470, 400], [767, 500], [767, 575], [0, 575]]
ZONE_PICK = {'points': None}

def pick_zone(img, max_width=768):
    """Colab 화면에서 클릭으로 다각형을 그리는 도구. 완료 버튼을 누르면 ZONE_PICK에 저장"""
    ok, buf = cv2.imencode('.jpg', img)
    b64 = base64.b64encode(buf).decode()
    html = '''
<canvas id="zc" style="cursor:crosshair;max-width:MAXWpx;width:100%"></canvas><br>
<button id="zundo">되돌리기</button> <button id="zdone">완료</button> <span id="zmsg"></span>
<script>
const img = new Image(); img.src = "data:image/jpeg;base64,B64";
const cv = document.getElementById("zc"), ctx = cv.getContext("2d"); let pts = [];
img.onload = () => { cv.width = img.width; cv.height = img.height; draw(); };
function draw() {
  ctx.drawImage(img, 0, 0);
  if (!pts.length) return;
  ctx.beginPath(); ctx.moveTo(pts[0][0], pts[0][1]);
  for (const p of pts.slice(1)) ctx.lineTo(p[0], p[1]);
  ctx.closePath(); ctx.fillStyle = "rgba(255,0,0,0.25)"; ctx.fill();
  ctx.strokeStyle = "red"; ctx.lineWidth = 2; ctx.stroke();
  for (const p of pts) { ctx.beginPath(); ctx.arc(p[0], p[1], 5, 0, 7); ctx.fillStyle = "yellow"; ctx.fill(); }
}
cv.onclick = (e) => {
  const r = cv.getBoundingClientRect();
  pts.push([Math.round((e.clientX - r.left) * cv.width / r.width), Math.round((e.clientY - r.top) * cv.height / r.height)]);
  draw();
};
document.getElementById("zundo").onclick = () => { pts.pop(); draw(); };
document.getElementById("zdone").onclick = async () => {
  if (pts.length < 3) { document.getElementById("zmsg").textContent = " 점을 3개 이상 찍으세요"; return; }
  await google.colab.kernel.invokeFunction("zone.set", [pts], {});
  document.getElementById("zmsg").textContent = " 저장됨: 점 " + pts.length + "개";
};
</script>'''.replace('B64', b64).replace('MAXW', str(max_width))
    display(HTML(html))

if IN_COLAB:
    output.register_callback('zone.set', lambda pts: ZONE_PICK.update(points=pts))
    pick_zone(FIRST)
else:
    print('Colab이 아니므로 그리기 도구를 건너뜁니다. 기본 구역을 사용합니다.')

In [ ]:
# 그린 구역이 있으면 그것을, 없으면 기본 구역을 사용
ZONE = np.array(ZONE_PICK['points'] or DEFAULT_ZONE, np.int32)
print('금지 구역 꼭짓점:', ZONE.tolist())

def draw_zone(img, zone, color=(0, 0, 255), alpha=0.25):
    overlay = img.copy()
    cv2.fillPoly(overlay, [zone], color)
    img = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)
    cv2.polylines(img, [zone], True, color, 2)
    return img

show_bgr(draw_zone(FIRST.copy(), ZONE), 'restricted zone')

### 3-2. 점이 다각형 안에 있는가

`cv2.pointPolygonTest(다각형, 점, False)`는 점이 다각형 **안이면 +1, 경계 위면 0, 밖이면 -1** 을 돌려줍니다. 발 위치가 0 이상이면 구역 안으로 봅니다.

In [ ]:
def in_zone(x, y, zone=None):
    zone = ZONE if zone is None else zone          # 호출할 때의 ZONE을 사용 (구역을 다시 그려도 반영)
    return cv2.pointPolygonTest(zone.reshape(-1, 1, 2).astype(np.float32), (float(x), float(y)), False) >= 0

# 검산: 구역의 무게중심(안쪽)과 화면 왼쪽 위 모서리(바깥)
print('구역 무게중심(안):', in_zone(*ZONE.mean(0)))
print('화면 왼쪽 위(밖):', in_zone(5, 5))

obs['in_zone'] = [in_zone(x, y) for x, y in zip(obs['foot_x'], obs['foot_y'])]
per_frame = obs.groupby('frame')['in_zone'].sum().reindex(range(N_FRAMES), fill_value=0)
plt.figure(figsize=(10, 3))
plt.fill_between(per_frame.index / FPS, per_frame.values, step='mid', color='tab:red', alpha=.5)
plt.xlabel('time (s)'); plt.ylabel('people in zone'); plt.title('People inside the restricted zone')
plt.grid(alpha=.3); plt.show()
print('구역 안 관측 비율:', round(obs['in_zone'].mean(), 3), '| 구역에 들어간 적 있는 ID:', obs.loc[obs['in_zone'], 'track_id'].nunique())

---
## Step 4. 진입·이탈 이벤트와 체류 시간 경고

'구역 안 프레임'을 그대로 경고로 쓰면 경계선에서 발이 왔다 갔다 할 때 경고가 깜빡입니다. 그래서 두 가지 규칙을 둡니다.

| 규칙 | 값 | 이유 |
|---|---|---|
| 진입 확정 | 구역 안에 **연속 `ENTER_FRAMES` 프레임** | 경계에서 한두 프레임 걸친 것은 무시 |
| 이탈 확정 | 구역 밖에 **연속 `EXIT_FRAMES` 프레임** (또는 추적이 끝남) | 잠깐 가려진 것을 이탈로 보지 않음 |
| 경고 | 체류 시간이 **`DWELL_WARN_S` 초 이상** | 지나가기만 한 사람과 머무는 사람 구분 |

한 사람이 들어갔다 나오는 한 번을 **이벤트 1건**으로 기록합니다.

In [ ]:
ENTER_FRAMES = 3
EXIT_FRAMES = 5
DWELL_WARN_S = 3.0

def zone_events(obs, fps=FPS):
    events = []
    for tid, g in obs.sort_values('frame').groupby('track_id'):
        inside, run_in, run_out, start = False, 0, 0, None
        frames, flags = g['frame'].to_numpy(), g['in_zone'].to_numpy()
        for f, flag in zip(frames, flags):
            if flag:
                run_in, run_out = run_in + 1, 0
                if not inside and run_in >= ENTER_FRAMES:
                    inside, start = True, f - ENTER_FRAMES + 1
            else:
                run_out, run_in = run_out + 1, 0
                if inside and run_out >= EXIT_FRAMES:
                    events.append((tid, start, f - EXIT_FRAMES)); inside = False
        if inside:
            events.append((tid, start, frames[-1]))
    rows = []
    for k, (tid, s, e) in enumerate(sorted(events, key=lambda x: x[1]), 1):
        dwell = (e - s + 1) / fps
        rows.append({'event_id': k, 'track_id': tid, 'enter_frame': int(s), 'exit_frame': int(e),
                     'enter_s': round(s / fps, 1), 'exit_s': round(e / fps, 1), 'dwell_s': round(dwell, 1),
                     'warning': int(dwell >= DWELL_WARN_S)})
    return pd.DataFrame(rows, columns=['event_id', 'track_id', 'enter_frame', 'exit_frame',
                                       'enter_s', 'exit_s', 'dwell_s', 'warning'])

# 검산 — 가상 관측: 2프레임만 걸친 사람(무시), 20프레임 머문 사람(이벤트 1건, 2.0초)
test = pd.DataFrame({'track_id': [1] * 10 + [2] * 30, 'frame': list(range(10)) + list(range(30)),
                     'in_zone': [0, 0, 1, 1, 0, 0, 0, 0, 0, 0] + [0] * 5 + [1] * 20 + [0] * 5})
ev = zone_events(test, fps=10)   # 검산은 10fps 기준으로 계산
print(ev)
assert len(ev) == 1 and ev.iloc[0]['track_id'] == 2 and ev.iloc[0]['dwell_s'] == 2.0
print('이벤트 규칙이 기대대로 동작합니다.')

In [ ]:
events = zone_events(obs)
events.to_csv('zone_events.csv', index=False); save_out('zone_events.csv')
print(f'침입 이벤트 {len(events)}건 | 경고(체류 {DWELL_WARN_S:.0f}초 이상) {int(events["warning"].sum())}건')
events

### 4-1. 감시 영상 만들기

관측 표와 이벤트 표로 영상을 다시 그립니다.
- 금지 구역: 아무도 없으면 **초록**, 누가 있으면 **주황**, 경고 대상이 있으면 **빨강**
- 사람 박스: 구역 밖은 흰색, 구역 안은 주황, 경고 대상은 빨강 + 머문 시간 표시
- 상단: 현재 구역 안 인원, 지금까지의 경고 건수

In [ ]:
def render_monitor(src, obs, events, dst='zone_monitor.mp4'):
    by_frame = {f: g for f, g in obs.groupby('frame')}
    cap = cv2.VideoCapture(src)
    writer = cv2.VideoWriter(dst, cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H))
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        active = events[(events['enter_frame'] <= i) & (events['exit_frame'] >= i)]
        dwell_now = {r.track_id: (i - r.enter_frame + 1) / FPS for r in active.itertuples()}
        warn_ids = {t for t, d in dwell_now.items() if d >= DWELL_WARN_S}
        zone_col = (0, 0, 255) if warn_ids else (0, 165, 255) if dwell_now else (0, 200, 0)
        frame = draw_zone(frame, ZONE, zone_col, 0.22)
        for r in by_frame.get(i, pd.DataFrame()).itertuples():
            col = (0, 0, 255) if r.track_id in warn_ids else (0, 165, 255) if r.track_id in dwell_now else (230, 230, 230)
            cv2.rectangle(frame, (int(r.x1), int(r.y1)), (int(r.x2), int(r.y2)), col, 2)
            cv2.circle(frame, (int(r.foot_x), int(r.foot_y)), 4, col, -1)
            label = f'#{r.track_id}' + (f' {dwell_now[r.track_id]:.1f}s' if r.track_id in dwell_now else '')
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
            cv2.rectangle(frame, (int(r.x1), int(r.y1) - th - 10), (int(r.x1) + tw + 4, int(r.y1) - 2), (0, 0, 0), -1)
            cv2.putText(frame, label, (int(r.x1) + 2, int(r.y1) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2)
        n_warn = int(((events['warning'] == 1) & (events['enter_frame'] + DWELL_WARN_S * FPS <= i)).sum())
        cv2.rectangle(frame, (0, 0), (W, 34), (0, 0, 0), -1)
        cv2.putText(frame, f'IN ZONE: {len(dwell_now)}   ALERTS: {n_warn}   t={i / FPS:5.1f}s', (10, 24),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        if warn_ids:
            cv2.putText(frame, 'WARNING: STAYING IN RESTRICTED ZONE', (10, H - 16),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 0, 255), 2)
        writer.write(frame)
        i += 1
    cap.release(); writer.release()
    return dst

render_monitor(VIDEO, obs, events)
save_out('zone_monitor.mp4')
show_video('zone_monitor.mp4')

**확인해 보기**
- 경고가 뜬 사람이 실제로 구역 안에 오래 있었나요? 영상과 `events` 표를 맞춰 보세요.
- `ENTER_FRAMES`를 1로, `EXIT_FRAMES`를 1로 바꾸면 이벤트 수가 어떻게 바뀌나요? 경계선을 스치듯 지나간 사람이 따로따로 여러 건으로 잡히는지 확인하세요.

---
## Step 5. 동선 히트맵과 감시 리포트

### 5-1. 동선 히트맵

모든 발 위치를 한 장에 쌓으면 사람들이 **실제로 많이 다니는 길**이 보입니다. 점마다 작은 가우시안(흐릿한 점)을 더해 부드러운 밀도 지도를 만듭니다.

In [ ]:
def foot_heatmap(obs, sigma=12):
    heat = np.zeros((H, W), np.float32)
    for x, y in zip(obs['foot_x'].astype(int).clip(0, W - 1), obs['foot_y'].astype(int).clip(0, H - 1)):
        heat[y, x] += 1
    heat = cv2.GaussianBlur(heat, (0, 0), sigma)
    return heat / heat.max()

heat = foot_heatmap(obs)
color = cv2.applyColorMap((heat * 255).astype(np.uint8), cv2.COLORMAP_JET)
mask = (heat > 0.03)[..., None]
heat_img = np.where(mask, cv2.addWeighted(FIRST, 0.45, color, 0.55, 0), FIRST)
cv2.polylines(heat_img, [ZONE], True, (255, 255, 255), 2)
cv2.imwrite('heatmap.png', heat_img); save_out('heatmap.png')
show_bgr(heat_img, 'Foot-traffic heatmap (white = restricted zone)')

**확인해 보기** — 히트맵에서 붉은 길이 금지 구역을 가로지르나요? 그렇다면 사람들이 그 구역을 **지름길**로 쓰고 있다는 뜻입니다. 경고만 늘릴 것이 아니라 동선을 바꾸거나 구역 경계를 옮겨야 할 수도 있습니다. 데이터로 현장 개선점을 찾는 것이 감시 시스템의 진짜 가치입니다.

### 5-2. 감시 리포트

In [ ]:
SUMMARY = {'video_s': round(N_FRAMES / FPS, 1), 'people_tracked': int(obs['track_id'].nunique()),
           'zone_events': len(events), 'warnings': int(events['warning'].sum()),
           'max_dwell_s': float(events['dwell_s'].max()) if len(events) else 0.0,
           'total_dwell_s': round(float(events['dwell_s'].sum()), 1) if len(events) else 0.0}
json.dump(SUMMARY, open('zone_summary.json', 'w'), indent=2); save_out('zone_summary.json')
display(pd.DataFrame([SUMMARY]))

fig = plt.figure(figsize=(12, 6.5))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.2])
ax = fig.add_subplot(gs[0, :])
ax.fill_between(per_frame.index / FPS, per_frame.values, step='mid', color='tab:orange', alpha=.4, label='people in zone')
for r in events.itertuples():
    ax.axvspan(r.enter_s, r.exit_s, ymin=0.9, ymax=1.0, color='red' if r.warning else 'gray', alpha=.6)
ax.set_xlim(0, N_FRAMES / FPS); ax.set_xlabel('time (s)'); ax.set_ylabel('people')
ax.set_title(f"Zone report | {SUMMARY['zone_events']} entries, {SUMMARY['warnings']} warnings, "
             f"longest stay {SUMMARY['max_dwell_s']} s  (top bars: red = warning)")
ax.legend(loc='upper left'); ax.grid(alpha=.3)
ax2 = fig.add_subplot(gs[1, 0])
ax2.hist(events['dwell_s'], bins=15, color='tab:red', alpha=.7)
ax2.axvline(DWELL_WARN_S, ls='--', c='k'); ax2.set_xlabel('dwell time (s)'); ax2.set_ylabel('entries')
ax2.set_title('How long people stayed')
ax3 = fig.add_subplot(gs[1, 1])
ax3.imshow(heat_img[..., ::-1]); ax3.axis('off'); ax3.set_title('Foot-traffic heatmap')
plt.tight_layout(); plt.savefig('zone_report.png', dpi=120); save_out('zone_report.png'); plt.show()

In [ ]:
obs_out = obs.round(2)
obs_out.to_csv('zone_frames.csv', index=False); save_out('zone_frames.csv')
print(sorted(p.name for p in SAVE_DIR.iterdir()))

결과 파일 (`SAVE_DIR`에 저장)

| 파일 | 내용 |
|---|---|
| `zone_frames.csv` | 프레임별 사람 관측 (`frame, time_s, track_id, x1, y1, x2, y2, foot_x, foot_y, conf, in_zone`) |
| `zone_events.csv` | 침입 이벤트 (`event_id, track_id, enter_frame, exit_frame, enter_s, exit_s, dwell_s, warning`) |
| `zone_monitor.mp4` | 감시 영상 |
| `heatmap.png` | 동선 히트맵 |
| `zone_report.png`, `zone_summary.json` | 감시 리포트 |

---
## 도전 과제

1. **구역 여러 개** — 도로(차량 통행 구역)와 잔디(장비 구역)를 따로 그리고, 구역마다 경고 시간을 다르게 설정해 보세요. `ZONE`을 딕셔너리로 바꾸면 됩니다.
2. **방향 규칙** — '나가는 방향으로만 지나가도 되는 통로'를 만들어 보세요. 발 위치의 이동 방향(`foot_x`의 변화)으로 역방향 진입을 찾습니다.
3. **혼잡 경고** — 구역 안 인원이 3명 이상이면 `CROWDED` 경고를 내는 규칙을 추가하세요.
4. **내 영상으로** — 공공데이터포털·캐글 등에서 받은 CCTV 영상이나 직접 찍은 영상으로 바꿔 실행해 보세요. `VIDEO` 경로만 바꾸면 됩니다. 카메라 높이에 따라 `CONF`와 `ENTER_FRAMES`를 다시 맞춰야 할 수 있습니다.

## 정리

- 탐지 단계에서 `classes=[0]`으로 **필요한 클래스만** 남기면 빠르고 단순합니다.
- 바닥 위 구역을 판정할 때는 박스 중심이 아니라 **발 위치**를 씁니다.
- 프레임 판정을 그대로 쓰지 않고 **연속 프레임 조건**으로 이벤트를 만들어야 경고가 깜빡이지 않습니다.
- 경고 기록만큼 중요한 것은 **동선 데이터**입니다. 히트맵으로 금지 구역 설계 자체를 개선할 근거를 얻습니다.